In [0]:
%sql
describe table intellibi_catalog.intellibi_gold.RefinedCustomer

In [0]:
dbutils.widgets.dropdown(
    name="Environment",
    defaultValue="dev",
    choices=["dev","qa","prd"],
    label="Select Environment"
)




In [0]:
env=dbutils.widgets.get("Environment")

catalog = f"saleslake_{env}"
schema = f"gold_{env}"
table_name = f"{catalog}.{schema}.RefinedCustomer"
print(table_name)

In [0]:
sql_query =f"""

MERGE INTO {table_name} AS target

USING (

    WITH latest_source AS (

        SELECT *
        FROM  saleslake_{env}.silver_{env}.cleanedCustomer

        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY ingest_ts DESC
        ) = 1

    ),

    comparison AS (

        SELECT

            source.*,

            gold.customer_sk,
            gold.rec_version AS previous_rec_version,

            CASE

                WHEN gold.customer_sk IS NULL
                    THEN 'NEW'

                WHEN NOT (
                    source.customer_name <=> gold.customer_name
                    AND source.email <=> gold.email
                    AND source.phone <=> gold.phone
                    AND source.address <=> gold.address
                    AND source.city <=> gold.city
                    AND source.state <=> gold.state
                    AND source.country <=> gold.country
                    AND source.zip_code <=> gold.zip_code
                    AND source.segment <=> gold.segment
                )
                    THEN 'CHANGE'

                ELSE 'NO_CHANGE'

            END AS record_status

        FROM latest_source source

        LEFT JOIN {table_name} gold

            ON CAST(source.customer_id AS INT) = gold.customer_id
            AND gold.end_effective_ts IS NULL

    ),

    actions AS (

        -- Expire existing record
        SELECT

            customer_sk AS merge_customer_sk,
            'EXPIRE' AS action,

            customer_id,
            customer_name,
            email,
            phone,
            address,
            city,
            state,
            country,
            zip_code,
            segment,
            ingest_ts,
            previous_rec_version

        FROM comparison

        WHERE record_status = 'CHANGE'


        UNION ALL


        -- Insert new customer or new version
        SELECT

            CAST(NULL AS BIGINT) AS merge_customer_sk,
            'INSERT' AS action,

            customer_id,
            customer_name,
            email,
            phone,
            address,
            city,
            state,
            country,
            zip_code,
            segment,
            ingest_ts,
            previous_rec_version

        FROM comparison

        WHERE record_status IN ('NEW', 'CHANGE')

    )

    SELECT *
    FROM actions

) AS source

ON target.customer_sk = source.merge_customer_sk
AND target.end_effective_ts IS NULL


-- Expire old version
WHEN MATCHED
AND source.action = 'EXPIRE'

THEN UPDATE SET

    target.isActive = 'False',
    target.end_effective_ts = source.ingest_ts


-- Insert new version
WHEN NOT MATCHED
AND source.action = 'INSERT'

THEN INSERT (

    customer_id,
    customer_name,
    email,
    phone,
    address,
    city,
    state,
    country,
    zip_code,
    segment,
    isActive,
    rec_version,
    start_effective_ts,
    end_effective_ts

)

VALUES (

    CAST(source.customer_id AS INT),
    source.customer_name,
    source.email,
    source.phone,
    source.address,
    source.city,
    source.state,
    source.country,
    source.zip_code,
    source.segment,
    'True',
    COALESCE(source.previous_rec_version, 0) + 1,
    source.ingest_ts,
    NULL

)"""

spark.sql(sql_query).show()

In [0]:
%sql
select * from saleslake_dev.gold_dev.RefinedCustomer order by customer_id